In [ ]:
!pip install pandas
!pip install bs4
!pip install waybackpack


In [ ]:
import pandas as pd

path = r"..\\archive\\Sarcasm_Headlines_Dataset_v2.json"
df = pd.read_json(path, lines=True)

print(df.head())
print(df.shape)
print(df.columns)


In [ ]:
import json
import re
import shutil
import subprocess
import threading
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import urlsplit, urlunsplit

import pandas as pd
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from tqdm import tqdm
from urllib3.util.retry import Retry

try:
    import waybackpack  # noqa: F401
except ImportError as e:
    raise ImportError("waybackpack is required. Install with: pip install waybackpack") from e

WAYBACKPACK_BIN = shutil.which("waybackpack")
if not WAYBACKPACK_BIN:
    raise RuntimeError("'waybackpack' command not found. Install with: pip install waybackpack")

SNAPSHOT_COLS = [
    "wayback_available",
    "wayback_url",
    "wayback_timestamp",
    "wayback_status",
    "wayback_error",
]

CONTENT_COLS = [
    "archived_title",
    "archived_meta_description",
    "archived_article_text",
    "content_error",
]

ALL_COLS = SNAPSHOT_COLS + CONTENT_COLS
_thread_local = threading.local()

DOMAIN_PATTERNS = [
    "https://www.huffingtonpost.com/*",
    "https://www.theonion.com/*",
    "https://local.theonion.com/*",
    "https://politics.theonion.com/*",
    "https://entertainment.theonion.com/*",
    "https://sports.theonion.com/*",
    "https://ogn.theonion.com/*",
]

TARGET_HOST_TOKENS = ("huffingtonpost.com", "theonion.com")


# -----------------------------
# Cache helpers
# -----------------------------
def _save_json(path: Path, obj: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=True)
    tmp.replace(path)

def _load_json(path: Path):
    if not path.exists():
        return {}
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        return data if isinstance(data, dict) else {}
    except Exception:
        return {}


# -----------------------------
# HTTP helpers
# -----------------------------
def build_session(retries: int = 5, backoff_factor: float = 1.0):
    s = requests.Session()
    retry = Retry(
        total=retries,
        connect=retries,
        read=retries,
        backoff_factor=backoff_factor,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry, pool_connections=50, pool_maxsize=50)
    s.mount("http://", adapter)
    s.mount("https://", adapter)
    s.headers.update({"User-Agent": "Mozilla/5.0"})
    return s

def get_session(retries: int = 5, backoff_factor: float = 1.0):
    if (
        not hasattr(_thread_local, "session")
        or getattr(_thread_local, "retries", None) != retries
        or getattr(_thread_local, "backoff_factor", None) != backoff_factor
    ):
        _thread_local.session = build_session(retries=retries, backoff_factor=backoff_factor)
        _thread_local.retries = retries
        _thread_local.backoff_factor = backoff_factor
    return _thread_local.session

def safe_get(url, params=None, timeout=(5, 25), retries: int = 5, backoff_factor: float = 1.0):
    r = get_session(retries=retries, backoff_factor=backoff_factor).get(url, params=params, timeout=timeout)
    r.raise_for_status()
    return r


# -----------------------------
# URL normalization helpers
# -----------------------------
def clean_text(text: str) -> str:
    return " ".join(text.split()) if text else ""

def normalize_dataset_url(url: str):
    if not isinstance(url, str):
        return None
    u = url.strip()
    if not u:
        return None

    # Fix malformed entries like: https://www.huffingtonpost.comhttp://...
    bad_prefixes = [
        "https://www.huffingtonpost.comhttp://",
        "https://www.huffingtonpost.comhttps://",
        "http://www.huffingtonpost.comhttp://",
        "http://www.huffingtonpost.comhttps://",
    ]
    for p in bad_prefixes:
        if u.startswith(p):
            u = u.replace("https://www.huffingtonpost.com", "", 1)
            u = u.replace("http://www.huffingtonpost.com", "", 1)
            break

    return u

def canonical_url_key(url: str):
    if not isinstance(url, str) or not url.strip():
        return None
    try:
        p = urlsplit(url.strip())
    except Exception:
        return None

    if not p.netloc:
        return None

    host = p.netloc.lower()
    path = p.path or "/"
    if path != "/" and path.endswith("/"):
        path = path[:-1]
    query = p.query or ""
    return urlunsplit(("https", host, path, query, ""))

def url_variants(url: str):
    if not isinstance(url, str) or not url.strip():
        return []
    u = url.strip()
    p = urlsplit(u)
    netloc = p.netloc.lower()
    path = p.path or "/"
    variants = set()
    for keep_query in [True, False]:
        q = p.query if keep_query else ""
        for scheme in ["https", "http"]:
            variants.add(urlunsplit((scheme, netloc, path, q, "")))
            alt = (path[:-1] or "/") if path.endswith("/") else (path + "/")
            variants.add(urlunsplit((scheme, netloc, alt, q, "")))
    for v in list(variants):
        pv = urlsplit(v)
        n = pv.netloc
        n2 = n[4:] if n.startswith("www.") else "www." + n
        variants.add(urlunsplit((pv.scheme, n2, pv.path, pv.query, "")))
    return [u] + [x for x in variants if x != u]

def verify_dataset_domains(df: pd.DataFrame, url_col: str = "article_link"):
    raw_hosts = {}
    normalized_hosts = {}
    non_target_examples = []

    for raw in df[url_col].dropna().astype(str):
        # Raw host
        try:
            raw_h = urlsplit(raw).netloc.lower()
            raw_hosts[raw_h] = raw_hosts.get(raw_h, 0) + 1
        except Exception:
            pass

        # Normalized host
        nu = normalize_dataset_url(raw)
        key = canonical_url_key(nu) if nu else None
        if not key:
            continue
        nh = urlsplit(key).netloc.lower()
        normalized_hosts[nh] = normalized_hosts.get(nh, 0) + 1

        if not any(tok in nh for tok in TARGET_HOST_TOKENS) and len(non_target_examples) < 10:
            non_target_examples.append(raw)

    only_target_after_normalization = all(any(tok in h for tok in TARGET_HOST_TOKENS) for h in normalized_hosts)
    return {
        "raw_hosts": dict(sorted(raw_hosts.items(), key=lambda x: x[1], reverse=True)),
        "normalized_hosts": dict(sorted(normalized_hosts.items(), key=lambda x: x[1], reverse=True)),
        "only_target_after_normalization": only_target_after_normalization,
        "non_target_examples": non_target_examples,
    }


# -----------------------------
# Wayback index helpers (domain pre-index)
# -----------------------------
def empty_snapshot():
    return {
        "wayback_available": False,
        "wayback_url": None,
        "wayback_timestamp": None,
        "wayback_status": None,
        "wayback_error": None,
    }

def empty_content():
    return {
        "archived_title": None,
        "archived_meta_description": None,
        "archived_article_text": None,
        "content_error": None,
    }

def parse_capture_url(capture_url: str):
    m = re.search(r"/web/(\\d{6,14})(?:[a-z]{2}_)?/(https?://.+)$", capture_url)
    if not m:
        return None, None, None
    ts = m.group(1)
    original = m.group(2)
    if len(ts) < 14:
        ts = ts.ljust(14, "0")
    status_m = re.search(r"/web/\\d{6,14}(?:[a-z]{2}_)?(\\d{3})", capture_url)
    status = status_m.group(1) if status_m else "200"
    return ts, original, status

def collect_latest_captures_for_pattern(pattern: str, latest_index: dict, all_article_links: dict):
    cmd = [
        WAYBACKPACK_BIN,
        pattern,
        "--list",
        "--progress",
        "--from-date", "2000",
        "--to-date", "2026",
    ]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=None, text=True)
    read_count = 0
    kept_count = 0

    for line in proc.stdout:
        ln = line.strip()
        if not ln or "web.archive.org/web/" not in ln:
            continue
        read_count += 1
        ts, original, status = parse_capture_url(ln)
        if not original:
            continue

        key = canonical_url_key(original)
        if not key:
            continue

        host = urlsplit(key).netloc.lower()
        if not any(tok in host for tok in TARGET_HOST_TOKENS):
            continue

        all_article_links[key] = 1

        cur = latest_index.get(key)
        if (cur is None) or ((ts or "") > (cur.get("wayback_timestamp") or "")):
            latest_index[key] = {
                "wayback_available": True,
                "wayback_url": ln,
                "wayback_timestamp": ts,
                "wayback_status": status,
                "wayback_error": None,
            }
            kept_count += 1

    proc.wait()

    if proc.returncode != 0:
        raise RuntimeError(f"waybackpack failed for {pattern} with exit code {proc.returncode}")

    return read_count, kept_count

def build_domain_latest_index(index_cache_path: Path, all_links_cache_path: Path, patterns=None, rebuild=False):
    patterns = patterns or DOMAIN_PATTERNS
    if index_cache_path.exists() and all_links_cache_path.exists() and not rebuild:
        cached_index = _load_json(index_cache_path)
        cached_all_links = _load_json(all_links_cache_path)
        if cached_index and cached_all_links:
            print(f"[INDEX] loaded existing latest index with {len(cached_index)} canonical URLs")
            print(f"[INDEX] loaded all article links inventory with {len(cached_all_links)} canonical URLs")
            return cached_index, cached_all_links

    latest_index = {}
    all_article_links = {}
    for pat in patterns:
        print(f"[INDEX] scanning pattern: {pat}")
        read_count, kept_count = collect_latest_captures_for_pattern(pat, latest_index, all_article_links)
        print(f"[INDEX] captures_read={read_count} updates={kept_count} unique_now={len(latest_index)}")

    _save_json(index_cache_path, latest_index)
    _save_json(all_links_cache_path, all_article_links)
    print(f"[INDEX] saved {len(latest_index)} canonical URLs to {index_cache_path}")
    print(f"[INDEX] saved {len(all_article_links)} canonical URLs to {all_links_cache_path}")
    return latest_index, all_article_links


# -----------------------------
# Content extraction
# -----------------------------
def extract_content(wayback_url: str, retries: int = 5, backoff_factor: float = 1.0, max_chars: int = 4000):
    out = empty_content()
    if not isinstance(wayback_url, str) or not wayback_url.strip():
        out["content_error"] = "Invalid/empty wayback_url"
        return out

    try:
        r = safe_get(wayback_url, timeout=(5, 40), retries=retries, backoff_factor=backoff_factor)
        soup = BeautifulSoup(r.text, "lxml")
        for tag in soup(["script", "style", "noscript", "header", "footer", "nav", "aside"]):
            tag.decompose()

        out["archived_title"] = clean_text(soup.title.get_text()) if soup.title else None

        meta_desc = ""
        m = soup.find("meta", attrs={"name": "description"})
        if m and m.get("content"):
            meta_desc = clean_text(m["content"])
        out["archived_meta_description"] = meta_desc or None

        selectors = ["article", "[role='main']", ".article", ".entry-content", ".post-content", ".content", "main"]
        candidates = []
        for sel in selectors:
            for node in soup.select(sel):
                txt = clean_text(node.get_text(" ", strip=True))
                if len(txt) > 200:
                    candidates.append(txt)

        if candidates:
            text = max(candidates, key=len)
        else:
            paras = [clean_text(p.get_text(" ", strip=True)) for p in soup.find_all("p")]
            paras = [p for p in paras if len(p) > 40]
            text = " ".join(paras[:20])

        out["archived_article_text"] = text[:max_chars] if text else None
        return out
    except Exception as e:
        out["content_error"] = f"{type(e).__name__}: {e}"
        return out


# -----------------------------
# Main pipeline (domain pre-index + dataset matching)
# -----------------------------
def build_snapshot_cache_by_matching(df: pd.DataFrame, latest_index: dict, url_col: str = "article_link"):
    snapshot_cache = {}
    unique_urls = [u for u in df[url_col].dropna().astype(str).unique().tolist() if u.strip()]

    matched = 0
    for raw in unique_urls:
        out = empty_snapshot()
        norm = normalize_dataset_url(raw)
        if not norm:
            out["wayback_error"] = "Invalid/empty URL"
            snapshot_cache[raw] = out
            continue

        keys_to_try = []
        for u in url_variants(norm):
            k = canonical_url_key(u)
            if k:
                keys_to_try.append(k)
        keys_to_try = list(dict.fromkeys(keys_to_try))

        found = None
        for k in keys_to_try:
            if k in latest_index:
                found = latest_index[k]
                break

        if found:
            snapshot_cache[raw] = dict(found)
            matched += 1
        else:
            snapshot_cache[raw] = out

    print(f"[MATCH] unique_urls={len(unique_urls)} matched={matched} unmatched={len(unique_urls)-matched}")
    return snapshot_cache

def enrich_with_wayback_domain_index(
    df: pd.DataFrame,
    url_col: str = "article_link",
    workers_content: int = 8,
    retries: int = 5,
    backoff_factor: float = 1.0,
    cache_dir: str = "./wayback_cache",
    checkpoint_csv: str = "headline_with_wayback_context.csv",
    save_every_n_urls: int = 500,
    rerun_errors: bool = True,
    rebuild_index: bool = False,
):
    if url_col not in df.columns:
        raise ValueError(f"DataFrame must contain '{url_col}'")

    cache_dir = Path(cache_dir)
    snapshot_cache_path = cache_dir / "snapshot_cache.json"
    content_cache_path = cache_dir / "content_cache.json"
    index_cache_path = cache_dir / "domain_latest_index.json"
    all_links_cache_path = cache_dir / "domain_all_article_links.json"

    domain_check = verify_dataset_domains(df, url_col=url_col)
    print("[VERIFY] raw hosts (top):", list(domain_check["raw_hosts"].items())[:10])
    print("[VERIFY] normalized hosts (top):", list(domain_check["normalized_hosts"].items())[:10])
    print("[VERIFY] only onion/huffpost after normalization:", domain_check["only_target_after_normalization"])
    if domain_check["non_target_examples"]:
        print("[VERIFY] sample non-target normalized entries:")
        for ex in domain_check["non_target_examples"][:5]:
            print(" -", ex)

    latest_index, all_article_links = build_domain_latest_index(
        index_cache_path=index_cache_path,
        all_links_cache_path=all_links_cache_path,
        rebuild=rebuild_index,
    )
    print(f"[INDEX] all article links stored: {len(all_article_links)}")
    snapshot_cache = build_snapshot_cache_by_matching(df, latest_index, url_col=url_col)
    _save_json(snapshot_cache_path, snapshot_cache)

    content_cache = _load_json(content_cache_path)
    hit_urls = [u for u, s in snapshot_cache.items() if s.get("wayback_available") and s.get("wayback_url")]
    content_todo = []
    for u in hit_urls:
        c = content_cache.get(u)
        if c is None:
            content_todo.append(u)
        elif rerun_errors and c.get("content_error"):
            content_todo.append(u)

    print(f"[INFO] hit_urls={len(hit_urls)}")
    print(f"[INFO] content_todo={len(content_todo)} (missing + error reruns)")

    content_ok = content_err = 0
    for i in range(0, len(content_todo), save_every_n_urls):
        chunk = content_todo[i:i + save_every_n_urls]

        def _extract_for_url(u):
            wb_url = snapshot_cache[u]["wayback_url"]
            return extract_content(wb_url, retries=retries, backoff_factor=backoff_factor)

        with ThreadPoolExecutor(max_workers=workers_content) as ex:
            fut_to_url = {ex.submit(_extract_for_url, u): u for u in chunk}
            pbar = tqdm(total=len(chunk), desc=f"content {i}:{i+len(chunk)}")

            for fut in as_completed(fut_to_url):
                u = fut_to_url[fut]
                try:
                    c = fut.result()
                except Exception as e:
                    c = empty_content()
                    c["content_error"] = f"{type(e).__name__}: {e}"

                content_cache[u] = c
                if c.get("content_error"):
                    content_err += 1
                else:
                    content_ok += 1

                pbar.update(1)
                pbar.set_postfix(ok=content_ok, err=content_err, refresh=False)

            pbar.close()

        _save_json(content_cache_path, content_cache)
        print(f"[CONTENT SAVE] {content_cache_path} | ok={content_ok} err={content_err}")

    rows = []
    for u in df[url_col].fillna("").astype(str).tolist():
        s = snapshot_cache.get(u, empty_snapshot())
        c = content_cache.get(u, empty_content()) if s.get("wayback_available") else empty_content()
        row = {k: s.get(k) for k in SNAPSHOT_COLS}
        row.update({k: c.get(k) for k in CONTENT_COLS})
        rows.append(row)

    enriched = pd.concat([df.reset_index(drop=True), pd.DataFrame(rows, columns=ALL_COLS)], axis=1)
    enriched.to_csv(checkpoint_csv, index=False)

    hits = int(enriched["wayback_available"].fillna(False).sum())
    misses = int((~enriched["wayback_available"].fillna(False)).sum())
    snap_errors = int(enriched["wayback_error"].notna().sum())
    content_errors = int(enriched["content_error"].notna().sum())

    print(f"[DONE] hits={hits} misses={misses} snapshot_errors={snap_errors} content_errors={content_errors}")
    print(f"[DONE] output={Path(checkpoint_csv).resolve()}")
    return enriched


In [ ]:
enriched_df = enrich_with_wayback_domain_index(
    df,
    url_col="article_link",
    workers_content=8,
    retries=5,
    backoff_factor=1.0,
    cache_dir="./wayback_cache",
    checkpoint_csv="headline_with_wayback_context.csv",
    save_every_n_urls=500,
    rerun_errors=False,
    rebuild_index=False,
)
